In [51]:
import re
from eppy.modeleditor import IDF

In [75]:
IDD_FILE   = r"C:\EnergyPlusV22-1-0\Energy+.idd"
INPUT_IDF  = r"C:\Users\yhw15\Documents\1947-RP\ASHRAE901_OfficeMedium_STD2022\ASHRAE901_OfficeMedium_STD2022_NewYork_V2210.idf"
OUTPUT_IDF = r"C:\Users\yhw15\Documents\1947-RP\STD2022_NewYork_ASHP_v4\ASHRAE901_OfficeMedium_STD2022_NewYork_ASHP_v4.idf"

In [76]:
IDF.setiddname(IDD_FILE)
idf = IDF(INPUT_IDF)
print("IDF loaded successfully")
AHU_LIST = ["bot", "mid", "top"]

IDF loaded successfully


In [77]:
eb=[c.Name for c in idf.idfobjects["CURVE:BIQUADRATIC"]]
ec=[c.Name for c in idf.idfobjects["CURVE:CUBIC"]]
eq=[c.Name for c in idf.idfobjects["CURVE:QUADRATIC"]]

def biq(name,c1,c2,c3,c4,c5,c6,xmin,xmax,ymin,ymax):
    if name in eb: return
    c=idf.newidfobject("CURVE:BIQUADRATIC")
    c.Name=name;c.Coefficient1_Constant=c1;c.Coefficient2_x=c2;c.Coefficient3_x2=c3
    c.Coefficient4_y=c4;c.Coefficient5_y2=c5;c.Coefficient6_xy=c6
    c.Minimum_Value_of_x=xmin;c.Maximum_Value_of_x=xmax
    c.Minimum_Value_of_y=ymin;c.Maximum_Value_of_y=ymax;print(f"  + {name}")
def cub(name,c1,c2,c3,c4,xmin,xmax):
    if name in ec: return
    c=idf.newidfobject("CURVE:CUBIC")
    c.Name=name;c.Coefficient1_Constant=c1;c.Coefficient2_x=c2
    c.Coefficient3_x2=c3;c.Coefficient4_x3=c4
    c.Minimum_Value_of_x=xmin;c.Maximum_Value_of_x=xmax;print(f"  + {name}")
def quad(name,c1,c2,c3,xmin,xmax):
    if name in eq: return
    c=idf.newidfobject("CURVE:QUADRATIC")
    c.Name=name;c.Coefficient1_Constant=c1;c.Coefficient2_x=c2;c.Coefficient3_x2=c3
    c.Minimum_Value_of_x=xmin;c.Maximum_Value_of_x=xmax;print(f"  + {name}")

biq("ASHP_HTG_CapFT",  0.876825,-0.002955,-0.000058,0.025335,0.000196,-0.000043,15.0,24.0,-20.0,20.0)
cub("ASHP_HTG_CapFF",  0.694045465,0.474207981,-0.180783882,0.012530446,0.5,1.5)
biq("ASHP_HTG_EIRFT",  0.704658,0.008767,0.000625,-0.009037,0.000738,-0.001025,15.0,24.0,-20.0,20.0)
cub("ASHP_HTG_EIRFF",  1.570774717,-0.914152018,0.036729136,0.306648166,0.5,1.5)
quad("ASHP_HTG_PLF",   0.85,0.15,0.0,0.0,1.0)
biq("ASHP_HTG_DefrostEIRFT",1.0,0.0,0.0,0.0,0.0,0.0,15.0,24.0,-20.0,20.0)

  + ASHP_HTG_CapFT
  + ASHP_HTG_CapFF
  + ASHP_HTG_EIRFT
  + ASHP_HTG_EIRFF
  + ASHP_HTG_PLF
  + ASHP_HTG_DefrostEIRFT


In [78]:
eo=[n.Name for n in idf.idfobjects["OUTDOORAIR:NODE"]]
for ahu in AHU_LIST:
    for nm in [f"PACU_VAV_{ahu} ASHP Condenser Inlet Node",
               f"PACU_VAV_{ahu} ASHP CLG Condenser Inlet Node"]:
        if nm not in eo:
            n=idf.newidfobject("OUTDOORAIR:NODE");n.Name=nm;print(f"  + {nm}")

  + PACU_VAV_bot ASHP Condenser Inlet Node
  + PACU_VAV_bot ASHP CLG Condenser Inlet Node
  + PACU_VAV_mid ASHP Condenser Inlet Node
  + PACU_VAV_mid ASHP CLG Condenser Inlet Node
  + PACU_VAV_top ASHP Condenser Inlet Node
  + PACU_VAV_top ASHP CLG Condenser Inlet Node


In [79]:
print("Step: Removing Coil:Heating:Fuel objects")
for ahu in AHU_LIST:
    nm=f"PACU_VAV_{ahu} Heating Coil"
    m=[c for c in idf.idfobjects["COIL:HEATING:FUEL"] if c.Name==nm]
    if m: idf.removeidfobject(m[0]);print(f"  - {nm}")

Step: Removing Coil:Heating:Fuel objects
  - PACU_VAV_bot Heating Coil
  - PACU_VAV_mid Heating Coil
  - PACU_VAV_top Heating Coil


In [80]:
print("Step: Removing CoilSystem:Cooling:DX objects")
for ahu in AHU_LIST:
    nm=f"PACU_VAV_{ahu} Cooling Coil"
    m=[c for c in idf.idfobjects["COILSYSTEM:COOLING:DX"] if c.Name==nm]
    if m: idf.removeidfobject(m[0]);print(f"  - {nm}")

Step: Removing CoilSystem:Cooling:DX objects
  - PACU_VAV_bot Cooling Coil
  - PACU_VAV_mid Cooling Coil
  - PACU_VAV_top Cooling Coil


In [81]:
print("Step: Creating Coil:Cooling:DX:SingleSpeed (COP=3.4)")
for ahu in AHU_LIST:
    us_name      = f"PACU_VAV_{ahu} Unitary ASHP"
    us_inlet     = f"PACU_VAV_{ahu}_OA-PACU_VAV_{ahu}_CoolCNode"
    clg_out_node = f"{us_name} Cooling Coil Air Outlet"

    name=f"PACU_VAV_{ahu} ASHP Cooling Coil"
    c=idf.newidfobject("COIL:COOLING:DX:SINGLESPEED")
    c.Name=name;c.Availability_Schedule_Name="Always_On"
    c.Gross_Rated_Total_Cooling_Capacity="AUTOSIZE"
    c.Gross_Rated_Sensible_Heat_Ratio="AUTOSIZE"
    c.Gross_Rated_Cooling_COP=3.4
    c.Rated_Air_Flow_Rate="AUTOSIZE"
    c.Air_Inlet_Node_Name  = us_inlet       
    c.Air_Outlet_Node_Name = clg_out_node  
    c.Total_Cooling_Capacity_Function_of_Temperature_Curve_Name   ="LgDXalt_CapFT"
    c.Total_Cooling_Capacity_Function_of_Flow_Fraction_Curve_Name ="LgDXalt_CapFF"
    c.Energy_Input_Ratio_Function_of_Temperature_Curve_Name       ="LgDXalt_EIRFT"
    c.Energy_Input_Ratio_Function_of_Flow_Fraction_Curve_Name     ="LgDXalt_EIRFF"
    c.Part_Load_Fraction_Correlation_Curve_Name                   ="LgDXalt_PLR"
    c.Condenser_Air_Inlet_Node_Name=f"PACU_VAV_{ahu} ASHP CLG Condenser Inlet Node"
    c.Condenser_Type="AirCooled"
    c.Crankcase_Heater_Capacity=0.0
    c.Maximum_Outdoor_DryBulb_Temperature_for_Crankcase_Heater_Operation=10.0
    print(f"  + {name}")

Step: Creating Coil:Cooling:DX:SingleSpeed (COP=3.4)
  + PACU_VAV_bot ASHP Cooling Coil
  + PACU_VAV_mid ASHP Cooling Coil
  + PACU_VAV_top ASHP Cooling Coil


In [82]:
print("Step: Creating Coil:Heating:DX:SingleSpeed (COP=3.2)")
for ahu in AHU_LIST:
    us_name      = f"PACU_VAV_{ahu} Unitary ASHP"
    clg_out_node = f"{us_name} Cooling Coil Air Outlet"
    htg_out_node = f"{us_name} Heating Coil Air Outlet"

    name=f"PACU_VAV_{ahu} ASHP Heating Coil"
    c=idf.newidfobject("COIL:HEATING:DX:SINGLESPEED")
    c.Name=name;c.Availability_Schedule_Name="Always_On"
    c.Gross_Rated_Heating_Capacity="AUTOSIZE";c.Gross_Rated_Heating_COP=3.2
    c.Rated_Air_Flow_Rate="AUTOSIZE"
    c.Air_Inlet_Node_Name  = clg_out_node  
    c.Air_Outlet_Node_Name = htg_out_node 
    c.Heating_Capacity_Function_of_Temperature_Curve_Name           ="ASHP_HTG_CapFT"
    c.Heating_Capacity_Function_of_Flow_Fraction_Curve_Name         ="ASHP_HTG_CapFF"
    c.Energy_Input_Ratio_Function_of_Temperature_Curve_Name         ="ASHP_HTG_EIRFT"
    c.Energy_Input_Ratio_Function_of_Flow_Fraction_Curve_Name       ="ASHP_HTG_EIRFF"
    c.Part_Load_Fraction_Correlation_Curve_Name                     ="ASHP_HTG_PLF"
    c.Defrost_Energy_Input_Ratio_Function_of_Temperature_Curve_Name ="ASHP_HTG_DefrostEIRFT"
    # c.Minimum_Outdoor_DryBulb_Temperature_for_Compressor_Operation  =-8.0
    c.Minimum_Outdoor_DryBulb_Temperature_for_Compressor_Operation  =-15.0
    c.Maximum_Outdoor_DryBulb_Temperature_for_Defrost_Operation     =5.0
    c.Crankcase_Heater_Capacity=0.0
    c.Maximum_Outdoor_DryBulb_Temperature_for_Crankcase_Heater_Operation=10.0
    c.Defrost_Strategy="ReverseCycle";c.Defrost_Control="Timed"
    c.Defrost_Time_Period_Fraction=0.058333;c.Resistive_Defrost_Heater_Capacity="AUTOSIZE"
    print(f"  + {name}")

Step: Creating Coil:Heating:DX:SingleSpeed (COP=3.2)
  + PACU_VAV_bot ASHP Heating Coil
  + PACU_VAV_mid ASHP Heating Coil
  + PACU_VAV_top ASHP Heating Coil


In [83]:
print("Step: Creating Backup Coil:Heating:Electric")
for ahu in AHU_LIST:
    us_name      = f"PACU_VAV_{ahu} Unitary ASHP"
    htg_out_node = f"{us_name} Heating Coil Air Outlet"
    us_outlet    = f"PACU_VAV_{ahu}_HeatC-PACU_VAV_{ahu} FanNode"

    name=f"PACU_VAV_{ahu} Backup Electric Coil"
    c=idf.newidfobject("COIL:HEATING:ELECTRIC")
    c.Name=name;c.Availability_Schedule_Name="Always_On"
    c.Efficiency=1.0;c.Nominal_Capacity="AUTOSIZE"
    c.Air_Inlet_Node_Name          = htg_out_node
    c.Air_Outlet_Node_Name         = us_outlet
    c.Temperature_Setpoint_Node_Name = us_outlet
    print(f"  + {name}")

Step: Creating Backup Coil:Heating:Electric
  + PACU_VAV_bot Backup Electric Coil
  + PACU_VAV_mid Backup Electric Coil
  + PACU_VAV_top Backup Electric Coil


In [84]:
print("Step: AirLoopHVAC:UnitarySystem")
for ahu in AHU_LIST:
    name      = f"PACU_VAV_{ahu} Unitary ASHP"
    us_inlet  = f"PACU_VAV_{ahu}_OA-PACU_VAV_{ahu}_CoolCNode"
    us_outlet = f"PACU_VAV_{ahu}_HeatC-PACU_VAV_{ahu} FanNode"

    u=idf.newidfobject("AIRLOOPHVAC:UNITARYSYSTEM")
    u.Name=name;u.Control_Type="SetPoint"
    u.Availability_Schedule_Name="Always_On"
    u.Air_Inlet_Node_Name=us_inlet;u.Air_Outlet_Node_Name=us_outlet
    u.Cooling_Coil_Object_Type="Coil:Cooling:DX:SingleSpeed"
    u.Cooling_Coil_Name=f"PACU_VAV_{ahu} ASHP Cooling Coil"
    u.Cooling_Supply_Air_Flow_Rate_Method="SupplyAirFlowRate"
    u.Cooling_Supply_Air_Flow_Rate="AUTOSIZE"
    u.Heating_Coil_Object_Type="Coil:Heating:DX:SingleSpeed"
    u.Heating_Coil_Name=f"PACU_VAV_{ahu} ASHP Heating Coil"
    u.Heating_Supply_Air_Flow_Rate_Method="SupplyAirFlowRate"
    u.Heating_Supply_Air_Flow_Rate="AUTOSIZE"
    u.Supplemental_Heating_Coil_Object_Type="Coil:Heating:Electric"
    u.Supplemental_Heating_Coil_Name=f"PACU_VAV_{ahu} Backup Electric Coil"
    u.Maximum_Supply_Air_Temperature=50.0
    u.Maximum_Outdoor_DryBulb_Temperature_for_Supplemental_Heater_Operation=-8.0
    u.No_Load_Supply_Air_Flow_Rate_Method="SupplyAirFlowRate"
    u.No_Load_Supply_Air_Flow_Rate="AUTOSIZE"
    print(f"  + {name}")

Step: AirLoopHVAC:UnitarySystem
  + PACU_VAV_bot Unitary ASHP
  + PACU_VAV_mid Unitary ASHP
  + PACU_VAV_top Unitary ASHP


In [85]:
for ahu in AHU_LIST:
    bname=f"PACU_VAV_{ahu} Air Loop Main Branch"
    m=[b for b in idf.idfobjects["BRANCH"] if b.Name==bname]
    if not m: continue
    b=m[0]
    b.Component_2_Object_Type="AirLoopHVAC:UnitarySystem"
    b.Component_2_Name=f"PACU_VAV_{ahu} Unitary ASHP"
    b.Component_2_Inlet_Node_Name=f"PACU_VAV_{ahu}_OA-PACU_VAV_{ahu}_CoolCNode"
    b.Component_2_Outlet_Node_Name=f"PACU_VAV_{ahu}_HeatC-PACU_VAV_{ahu} FanNode"
    b.Component_3_Object_Type="Fan:VariableVolume"
    b.Component_3_Name=f"PACU_VAV_{ahu} Fan"
    b.Component_3_Inlet_Node_Name=f"PACU_VAV_{ahu}_HeatC-PACU_VAV_{ahu} FanNode"
    b.Component_3_Outlet_Node_Name=f"PACU_VAV_{ahu} Supply Equipment Outlet Node"
    b.Component_4_Object_Type=""  
    b.Component_4_Name=""
    b.Component_4_Inlet_Node_Name=""
    b.Component_4_Outlet_Node_Name=""
    print(f"  ✓ {bname}")

  ✓ PACU_VAV_bot Air Loop Main Branch
  ✓ PACU_VAV_mid Air Loop Main Branch
  ✓ PACU_VAV_top Air Loop Main Branch


In [86]:
idf.saveas(OUTPUT_IDF)

with open(OUTPUT_IDF,"r") as f: content=f.read()

pattern=(
    r"([ \t]+PACU_VAV_\w+ Supply Equipment Outlet Node)"
    r",[ \t]*!- Component 3 Outlet Node Name\n"
    r"[ \t]*,[ \t]*!- Component 4 Object Type\n"
    r"[ \t]*,[ \t]*!- Component 4 Name\n"
    r"[ \t]*,[ \t]*!- Component 4 Inlet Node Name\n"
    r"[ \t]*;[ \t]*!- Component 4 Outlet Node Name\n"
)
content=re.sub(pattern, lambda m: m.group(1)+";  !- Component 3 Outlet Node Name\n", content)
remaining=content.count("Component 4 Object Type")


with open(OUTPUT_IDF,"w") as f: f.write(content)
print(f"\n{'='*55}\n✓ {OUTPUT_IDF}\n{'='*55}")


✓ C:\Users\yhw15\Documents\1947-RP\STD2022_NewYork_ASHP_v4\ASHRAE901_OfficeMedium_STD2022_NewYork_ASHP_v4.idf
